# SQL Analysis — Churn Intelligence Queries

## Objective
Replicate key findings from Python analysis using SQL,
demonstrating ability to work with data in a real query environment.

## Queries Covered
1. Overall churn rate
2. Churn by contract type
3. Cohort retention analysis
4. Risk segment summary
5. Revenue at risk

In [1]:
import duckdb
import pandas as pd

# Connect to DuckDB and point to our CSV
con = duckdb.connect()

# Test connection
result = con.execute("""
    SELECT COUNT(*) as total_customers
    FROM read_csv_auto('../data/churn_scored.csv')
""").df()

print("✅ DuckDB connected!")
print(result)

✅ DuckDB connected!
   total_customers
0             7043


In [2]:
# Query 1 — Overall Churn Rate
q1 = con.execute("""
    SELECT 
        COUNT(*) as total_customers,
        SUM(Churn_Binary) as churned_customers,
        ROUND(AVG(Churn_Binary) * 100, 1) as churn_rate_pct,
        ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges
    FROM read_csv_auto('../data/churn_scored.csv')
""").df()

print("=== Query 1: Overall Churn Rate ===")
print(q1.to_string(index=False))

=== Query 1: Overall Churn Rate ===
 total_customers  churned_customers  churn_rate_pct  avg_monthly_charges
            7043             1869.0            26.5                64.76


In [3]:
# Query 2 — Churn by Contract Type
q2 = con.execute("""
    SELECT 
        Contract,
        COUNT(*) as total_customers,
        SUM(Churn_Binary) as churned,
        ROUND(AVG(Churn_Binary) * 100, 1) as churn_rate_pct,
        ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges
    FROM read_csv_auto('../data/churn_scored.csv')
    GROUP BY Contract
    ORDER BY churn_rate_pct DESC
""").df()

print("=== Query 2: Churn by Contract Type ===")
print(q2.to_string(index=False))

=== Query 2: Churn by Contract Type ===
      Contract  total_customers  churned  churn_rate_pct  avg_monthly_charges
Month-to-month             3875   1655.0            42.7                66.40
      One year             1473    166.0            11.3                65.05
      Two year             1695     48.0             2.8                60.77


In [4]:
# Query 3 — Cohort Retention Analysis (Window Functions)
q3 = con.execute("""
    WITH cohort_stats AS (
        SELECT 
            Tenure_Cohort,
            COUNT(*) as total_customers,
            SUM(Churn_Binary) as churned,
            ROUND(AVG(Churn_Binary) * 100, 1) as churn_rate_pct,
            ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges
        FROM read_csv_auto('../data/churn_scored.csv')
        GROUP BY Tenure_Cohort
    )
    SELECT 
        Tenure_Cohort,
        total_customers,
        churned,
        churn_rate_pct,
        avg_monthly_charges,
        ROUND(100.0 * total_customers / SUM(total_customers) OVER (), 1) as pct_of_total
    FROM cohort_stats
    ORDER BY churn_rate_pct DESC
""").df()

print("=== Query 3: Cohort Retention Analysis ===")
print(q3.to_string(index=False))

=== Query 3: Cohort Retention Analysis ===
Tenure_Cohort  total_customers  churned  churn_rate_pct  avg_monthly_charges  pct_of_total
  0-12 Months             2186   1037.0            47.4                56.10          31.0
 13-24 Months             1024    294.0            28.7                61.36          14.5
 25-48 Months             1594    325.0            20.4                65.93          22.6
   49+ Months             2239    213.0             9.5                73.95          31.8


In [5]:
# Query 4 — Risk Segment Revenue Analysis
q4 = con.execute("""
    SELECT 
        Risk_Segment,
        COUNT(*) as total_customers,
        ROUND(AVG(Churn_Binary) * 100, 1) as churn_rate_pct,
        ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges,
        ROUND(SUM(MonthlyCharges), 2) as total_monthly_revenue,
        ROUND(SUM(MonthlyCharges) * AVG(Churn_Binary), 2) as monthly_revenue_at_risk,
        ROUND(SUM(MonthlyCharges) * AVG(Churn_Binary) * 12, 2) as annual_revenue_at_risk
    FROM read_csv_auto('../data/churn_scored.csv')
    GROUP BY Risk_Segment
    ORDER BY churn_rate_pct DESC
""").df()

print("=== Query 4: Risk Segment Revenue Analysis ===")
print(q4.to_string(index=False))

=== Query 4: Risk Segment Revenue Analysis ===
Risk_Segment  total_customers  churn_rate_pct  avg_monthly_charges  total_monthly_revenue  monthly_revenue_at_risk  annual_revenue_at_risk
   High Risk             3132            48.9                72.02              225562.00                110404.39              1324852.67
 Medium Risk             1828            14.9                63.78              116592.65                 17348.58               208182.94
    Low Risk             2083             3.1                54.71              113961.95                  3501.47                42017.66


In [6]:
# Query 5 — High Risk Customer Profile
q5 = con.execute("""
    WITH high_risk AS (
        SELECT *
        FROM read_csv_auto('../data/churn_scored.csv')
        WHERE Risk_Segment = 'High Risk'
    ),
    low_risk AS (
        SELECT *
        FROM read_csv_auto('../data/churn_scored.csv')
        WHERE Risk_Segment = 'Low Risk'
    )
    SELECT 
        'High Risk' as Segment,
        ROUND(AVG(tenure), 1) as avg_tenure_months,
        ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges,
        ROUND(AVG(Churn_Binary) * 100, 1) as churn_rate_pct,
        COUNT(*) as customer_count
    FROM high_risk
    UNION ALL
    SELECT 
        'Low Risk' as Segment,
        ROUND(AVG(tenure), 1) as avg_tenure_months,
        ROUND(AVG(MonthlyCharges), 2) as avg_monthly_charges,
        ROUND(AVG(Churn_Binary) * 100, 1) as churn_rate_pct,
        COUNT(*) as customer_count
    FROM low_risk
""").df()

print("=== Query 5: High Risk vs Low Risk Customer Profile ===")
print(q5.to_string(index=False))

=== Query 5: High Risk vs Low Risk Customer Profile ===
  Segment  avg_tenure_months  avg_monthly_charges  churn_rate_pct  customer_count
High Risk               15.9                72.02            48.9            3132
 Low Risk               54.3                54.71             3.1            2083
